# Choosing the Optimal Data Management Architecture

## Background

In this exercise, you act as a data engineer tasked with recommending the most cost-efficient data management architecture for a company.

**Group competition (in class).** Each group calculates the 2-year costs for **one** architecture, the one your group explained in the expert round. The comparison table is assembled on the whiteboard from all groups. Afterwards every group gives a recommendation, with one rule: **you may not recommend your own architecture.**

At home: calculate all four and check the whiteboard numbers (bonus below).

**Background.** The company generates and uses data in various ways, and the decision must be made considering only the costs of storage, data transfer, querying, maintenance/administration, and data source acquisition, along with the frequency of these operations. In this exercise, we focus only on costs.
In real-life scenarios, cost is not the only factor and trade-offs have to be made.

The company is deciding between:
1. Data Warehouse
2.	Data Lake
3.	Hybrid Data Lake
4.	Lakehouse

Each architecture has different cost structures and performance trade-offs.

In [ ]:
# Your group's architecture: "Data Warehouse", "Data Lake", "Hybrid Data Lake" or "Lakehouse"
MY_ARCHITECTURE = "Data Lake"

In the following pandas dataframe the costs for the here considered operations are provided.

In [64]:
import pandas as pd

costs_information_dict = {
    "architecture": ["Data Warehouse", "Data Lake", "Hybrid Data Lake", "Lakehouse"],
    "Storage Cost ($/TB/month)": [300, 50, 350, 80],
    "Transfer Cost ($/GB)": [0.1,0.1,0.1,0.1],
    "Query Cost ($/1,000 queries)": [0.5, 1, 0.5, 0.75],
    "Maintenance Cost ($/month)": [50, 30, 70, 30],
}

costs = pd.DataFrame(costs_information_dict)
costs = costs.set_index('architecture')

costs

,Storage Cost ($/TB/month),Transfer Cost ($/GB),"Query Cost ($/1,000 queries)",Maintenance Cost ($/month)
architecture,,,,
Data Warehouse,300,0.1,0.50,50
Data Lake,50,0.1,1.00,30
Hybrid Data Lake,350,0.1,0.50,70
Lakehouse,80,0.1,0.75,30


## Usage Information

1. **Data sources** (acquisition cost per GB, paid to the source; plus transfer cost per GB, see cost table)
    - Source A: produces 10 GB of data **daily**, acquisition cost $0.2/GB
    - Source B: produces 100 GB of data **weekly**, acquisition cost $0.1/GB
    - Source C: produces 20 GB of data **monthly**, acquisition cost $1/GB
2. **Data storage** (storage cost per GB stored, billed every month)
    - Storage starts empty and grows by **20 GB per month**, data is kept for the whole 2 years
    - so month 1 bills 20 GB, month 2 bills 40 GB, ..., month 24 bills 480 GB
3. **Query patterns** (query cost per 1,000 queries)
    - Type 1 queries: run 1,000 times per **day**
    - Type 2 queries: run 10,000 times per **week**
    - Type 3 queries: run 15,000 times per **month**
4. **Maintenance**: fixed cost per month, see cost table

Use 1 year = 365 days = 52 weeks (so a week is 365/7 days and a month is 365/12 days for the daily conversions, or simply work per day: 2 years = 730 days).

## Task

### 1. Calculate the 2-year costs of your architecture (10 min)

For `MY_ARCHITECTURE`, calculate the total cost within 2 years of

- data source acquisition
- data storage
- data transfer
- querying the data
- maintenance/administration

and the overall sum. Print one line like `Data Lake: 1234 $ in 2 years (acquisition ..., storage ..., transfer ..., queries ..., maintenance ...)`.

### 2. Whiteboard

Write your overall number and the five components on the whiteboard. Compare with the other groups: same assumptions, same numbers?

### 3. Recommendation (2 min per group)

Based on the whiteboard table: which architecture would you recommend to the company? **Not your own.** Give one cost argument and one non-cost argument (performance, flexibility, risk of a data swamp, team skills, ...).

In [ ]:
costs["Storage Cost ($/GB/month)"] = costs["Storage Cost ($/TB/month)"] / 1000

DAYS = 2 * 365

# usage that is the same for every architecture
gb_per_day = 10 + 100 / 7 + 20 / (365 / 12)                       # from the three sources
acquisition_per_day = 10 * 0.2 + 100 / 7 * 0.1 + 20 / (365 / 12) * 1
transfer_gb = gb_per_day * DAYS
storage_gb_months = 20 * sum(range(1, 25))                        # 20 + 40 + ... + 480 = 6000
queries_count = DAYS * (1000 + 10000 / 7 + 15000 / (365 / 12))


def two_year_costs(architecture):
    c = costs.loc[architecture]
    acquisition = acquisition_per_day * DAYS
    storage = c["Storage Cost ($/GB/month)"] * storage_gb_months
    transfer = c["Transfer Cost ($/GB)"] * transfer_gb
    queries = c["Query Cost ($/1,000 queries)"] * queries_count / 1000
    maintenance = c["Maintenance Cost ($/month)"] * 24
    return {"acquisition": acquisition, "storage": storage, "transfer": transfer,
            "queries": queries, "maintenance": maintenance,
            "total": acquisition + storage + transfer + queries + maintenance}


mine = two_year_costs(MY_ARCHITECTURE)
print(f"{MY_ARCHITECTURE}: {mine['total']:.0f} $ in 2 years (acquisition {mine['acquisition']:.0f}, "
      f"storage {mine['storage']:.0f}, transfer {mine['transfer']:.0f}, queries {mine['queries']:.0f}, "
      f"maintenance {mine['maintenance']:.0f})")

## At home: all four architectures

Loop over all architectures, build one table (rows = architectures, columns = the five cost components + total) and sort it by total. Compare with the whiteboard photo.

In [ ]:
# all four, this is what the whiteboard table should look like
comparison = pd.DataFrame({a: two_year_costs(a) for a in costs.index}).T.round(0)
comparison.sort_values("total")

### Recommendation (example)

Within 2 years the **Lakehouse** is the cheapest option, mainly because cheap object storage dominates and the query engine is only moderately more expensive than a warehouse. The plain **Data Lake** is close behind on cost, but every query is more expensive and there is no built-in query execution: the risk of ending up with a data swamp is real if nobody owns the structure. The **Data Warehouse** costs about 15 % more and buys fast, governed queries, which matters if the main consumers are dashboards with many users. The **Hybrid Data Lake** is the most expensive because data is stored twice and two systems have to be maintained; it is only advisable if low-latency operational BI *and* cheap exploration are both hard requirements.

Beyond cost: team skills (who can run a query engine on object storage?), the share of unstructured data (images, documents for AI), and how often schemas change (warehouse backfills are expensive) decide the case in practice.

# Bonus Exercise

Assume after 6 month there was an incident. You lost all your _mart data_ due to an error in the transformation process. Unfortunately, you did not design a backup system. But it was a lesson for you, now you add a backup system, which means you add a copy of your database, a so called RAID I system (just assume a doubled storage price).

Additionally you realized after one year, you need to make a major update of the schema in the data warehouse, to fulfill all requirements. For this, you need all the raw data from the beginning which is not part of the actual data marts. In all the misfortune, you are somewhat lucky because the 'old' raw data is still available in the sources, i.e. for source A you have not only access to the 10GB which is produced on a daily basis but you also can access the data of the last 6 months and re-load it.

## Task

Calculate the costs for 2 years (take the events into account) for the different architectures. Welche Architektur ist jetzt stark im Preis gestiegen und hat keine guten